<p style="font-size:16pt; font-weight:bold; color:red; padding-bottom:20px; float:right">Please rename this file before editing!</p>

# Session 13 — A Worked Example: 97.5 % Accurate and Useless

Notebook 01 built a model on made-up coffee days. This notebook builds one on
real data and runs into the things that go wrong on real data, in the order
they usually go wrong.

**The data.** The consumer complaints of Session 9: 1,282,355 complaints about
financial products, sent on to companies by a U.S. regulator. Each one records
whether the company answered **on time**.

**The task.** A complaint arrives. Before it is sent on, can we tell whether
it will be answered **late**? An agency with limited staff could then chase
those companies first.

| Step | What happens |
|---|---|
| 1 | Load, and define the target |
| 2 | Split, and compute the baseline |
| 3 | A first model, with a spectacular accuracy |
| 4 | What the accuracy hid |
| 5 | A model that scores perfectly, and why it must be thrown away |
| 6 | A better feature, built without looking at the test set |
| 7 | Choosing the depth on a validation set |
| 8 | Will it work next year? |
| 9 | What to report |

> **Run this on ARC, with the kernel set to Conda Python 3.12.** The file is
> under `/data/public/` and exists only there. The notebook takes about twenty
> seconds.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, export_text

pd.options.display.float_format = "{:,.4f}".format

PATH = "/data/public/ConsumerComplaintDatabase/rows.csv"
SEED = 13

---

## Step 1 — Load, and define the target

In [ ]:
complaints = pd.read_csv(PATH, usecols=[
    "Date received", "Product", "Company", "Submitted via",
    "Company response to consumer", "Timely response?"])
complaints.columns = ["received", "product", "company", "via", "response", "timely"]
complaints["received"] = pd.to_datetime(complaints["received"], format="%m/%d/%Y")
complaints["year"] = complaints["received"].dt.year
complaints["month"] = complaints["received"].dt.month

print(complaints.shape)
complaints.head(3)

The target is **late**: 1 when the company did not answer on time. As in
Homework 12, the rare and interesting outcome is the one called 1.

In [ ]:
complaints["late"] = (complaints["timely"] == "No").astype(int)

print(complaints["late"].value_counts())
print("late: {:.2%}".format(complaints["late"].mean()))

One complaint in forty is answered late. The two classes are badly out of
balance, and everything in this notebook follows from that.

---

## Step 2 — Split, and compute the baseline

Features first. What is known when a complaint arrives: the product it is
about, how it was submitted, and the date. The product names changed in 2017
(Session 9 found that the hard way), so they are grouped; text columns become
0/1 columns.

In [ ]:
GROUPS = {
    "Credit reporting": "credit",
    "Credit reporting, credit repair services, or other personal consumer reports": "credit",
    "Mortgage": "mortgage",
    "Debt collection": "debt",
}
complaints["group"] = complaints["product"].map(GROUPS).fillna("other")

X = pd.concat([
    pd.get_dummies(complaints["via"], prefix="via").astype(int),
    pd.get_dummies(complaints["group"], prefix="product").astype(int),
    complaints[["year", "month"]],
], axis=1)
y = complaints["late"]

print(X.shape)
list(X.columns)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=SEED)

baseline = (y_test == 0).mean()
print("{:,} complaints to learn from, {:,} put aside".format(len(X_train), len(X_test)))
print("baseline, always 'on time': {:.4f}".format(baseline))

The baseline is 97.5 %. Answer "on time" for every complaint without reading
it and you are right 39 times in 40. Hold on to that number.

---

## Step 3 — A first model

In [ ]:
first = DecisionTreeClassifier(max_depth=8, random_state=SEED)
first.fit(X_train, y_train)

print("accuracy on the test set: {:.4f}".format(first.score(X_test, y_test)))

97.5 % accurate, on complaints the model has never seen. In most reports that
sentence would be the headline, and the work would stop here.

Compare it with the baseline of Step 2.

In [ ]:
print("model:    {:.4f}".format(first.score(X_test, y_test)))
print("baseline: {:.4f}".format(baseline))

---

## Step 4 — What the accuracy hid

The same. To four decimals, the model is exactly as good as not having one.
What is it doing?

In [ ]:
def confusion(actual, predicted):
    actual, predicted = np.asarray(actual), np.asarray(predicted)
    return {"tp": int(((predicted == 1) & (actual == 1)).sum()),
            "fp": int(((predicted == 1) & (actual == 0)).sum()),
            "fn": int(((predicted == 0) & (actual == 1)).sum()),
            "tn": int(((predicted == 0) & (actual == 0)).sum())}


def scores(actual, predicted):
    c = confusion(actual, predicted)
    total = sum(c.values())
    return {"accuracy": (c["tp"] + c["tn"]) / total,
            "precision": c["tp"] / (c["tp"] + c["fp"]) if c["tp"] + c["fp"] else np.nan,
            "recall": c["tp"] / (c["tp"] + c["fn"]) if c["tp"] + c["fn"] else np.nan,
            "called late": c["tp"] + c["fp"]}


predictions = first.predict(X_test)
print(confusion(y_test, predictions))
print(scores(y_test, predictions))

Out of 320,589 test complaints it called one late, and that one was not. It has
learned to say "on time", like the baseline. Its recall is zero: of the 8,098
late answers in the test set it found none.

**It is 97.5 % accurate and it does not do the job.** The job was to find the
late ones.

This is the trap of an unbalanced target. Accuracy rewards agreeing with the
majority, and when the majority is 39 in 40, the cheapest way to be accurate
is to never predict the thing you were asked to find. With a rare target,
accuracy is the wrong score. The scores for the job are recall (how many late
answers are found) and precision (how many alarms are real).

The features are the deeper problem. Product, channel and date say very little
about whether an answer will be late:

In [ ]:
print(complaints.groupby("via")["late"].mean().sort_values())
print()
print(complaints.groupby("group")["late"].mean().sort_values())

Between 1 % and 6 % whatever you group by. No question about these columns
splits off a group that is *mostly* late, so the tree never has a reason to
say "late". What would tell them apart?

---

## Step 5 — A model that scores perfectly

There is a column not used yet: the company's response.

In [ ]:
complaints["response"].value_counts(dropna=False)

In [ ]:
X_leaky = pd.concat([X, pd.get_dummies(complaints["response"], prefix="response").astype(int)], axis=1)

leaky = DecisionTreeClassifier(max_depth=3, random_state=SEED)
leaky.fit(X_leaky.loc[X_train.index], y_train)

result = scores(y_test, leaky.predict(X_leaky.loc[X_test.index]))
print({name: round(value, 4) for name, value in result.items()})

Precision close to 1.0: when this model says "late", it is right nearly every
time, and it finds a fifth of the late answers. A minute ago nothing worked.
Look at what the tree asks.

In [ ]:
print(export_text(leaky, feature_names=list(X_leaky.columns), max_depth=2))

Its first question is whether the response is **"Untimely response"**. That
category is written into the record *after* the company has answered late. It
is the outcome under another name.

This is **leakage**: a feature that would not exist at the moment the
prediction is needed. When the complaint arrives, nobody knows the company's
response yet. The model is useless on the day it is meant to be used, and its
excellent score describes its ability to read the answer off the page.

Leakage is the most common way for a model to look good and be worthless, and
it rarely announces itself this clearly. The habits that catch it:

- **A score that is too good is a warning.** Go and look.
- **Print the tree** (or whatever shows what the model uses), and read it.
- For every feature, ask: **would I have this value at the time of
  prediction?** A column filled in later, a total that includes the row
  itself, an id that encodes the date: all leaks.

The column goes. Back to honest features.

---

## Step 6 — A better feature, built without the test set

What would a person look at? **Who the complaint is about.** Some companies
answer everything on time; a few answer almost nothing on time (Session 9
found one with 369 complaints and no timely answer).

So: for each company, its share of late answers so far. That is a legitimate
thing to know when a new complaint arrives, as long as "so far" means what it
says. The share must be computed from the **training** complaints only. If
the test complaints were included, each test row would have contributed to its
own feature: a quieter leak than Step 5, and the same mistake.

In [ ]:
train_rows = complaints.loc[X_train.index]

history = train_rows.groupby("company")["late"].agg(late_share="mean", complaints="size")
overall = train_rows["late"].mean()

print(len(history), "companies seen in the training set")
history.sort_values("complaints", ascending=False).head()

In [ ]:
def with_history(features):
    """Add each complaint's company history, taken from the TRAINING set."""
    company = complaints.loc[features.index, "company"]
    added = features.copy()
    # A company never seen in training has no history: use the overall share.
    added["company_late_share"] = company.map(history["late_share"]).fillna(overall).to_numpy()
    added["company_complaints"] = company.map(history["complaints"]).fillna(0).to_numpy()
    return added


X_train_2 = with_history(X_train)
X_test_2 = with_history(X_test)
print("test complaints about a company never seen in training: {:,}".format(
    int((X_test_2["company_complaints"] == 0).sum())))

Two decisions are in that function, and both are the kind that Homework 12
asks you to state.

- **A missing value.** A company that is new in the test set has no history.
  It gets the overall late share, which is the honest "we do not know".
- **The same recipe for both parts.** The history table is built once, from
  training rows, and then *applied* to training and test rows alike.

In [ ]:
second = DecisionTreeClassifier(max_depth=3, random_state=SEED)
second.fit(X_train_2, y_train)

print("without history:", {k: round(v, 4) for k, v in scores(y_test, first.predict(X_test)).items()})
print("with history:   ", {k: round(v, 4) for k, v in scores(y_test, second.predict(X_test_2)).items()})

Accuracy moved from 97.5 % to 97.9 %, which sounds like nothing. Look at the
other columns. The model now raises about 3,300 alarms, 70 % of them real, and
finds 29 % of the late answers, where before it found none.

That is the lesson of Step 4 run in reverse: a change that barely moves the
accuracy turned a model that does nothing into one that does something.

In [ ]:
print(export_text(second, feature_names=list(X_train_2.columns)))

Every question is about the company's history. The tree says: a complaint
about a company that has answered more than about a third of its complaints
late will probably be answered late. Product, channel and date did not make it
into a tree of depth 3 at all.

That is not deep. It is also the right answer, it can be explained to anyone,
and it was found by thinking about the problem, not by a cleverer algorithm.
Most of the gain in machine learning comes from the features.

**✏️ Try it:** Which companies does the model flag? Take the test rows it calls late and count them by company (complaints.loc[...,'company'].value_counts().head()).

---

## Step 7 — Choosing the depth

Depth 3 was a guess. The honest way to choose is a validation set carved out
of the training set. One subtlety: the company history must then be built
from the *fit* part only, or the validation rows leak into their own feature.
To keep this notebook short, the history from the whole training set is
reused here, and the choice is therefore a little optimistic; exercise 3 does
it strictly.

In [ ]:
X_fit, X_valid, y_fit, y_valid = train_test_split(X_train_2, y_train, test_size=0.25, random_state=SEED)

rows = []
for depth in (1, 2, 3, 4, 6, 8, 12, 20):
    tree = DecisionTreeClassifier(max_depth=depth, random_state=SEED).fit(X_fit, y_fit)
    fit_scores = scores(y_fit, tree.predict(X_fit))
    valid_scores = scores(y_valid, tree.predict(X_valid))
    rows.append({"depth": depth, "leaves": tree.get_n_leaves(),
                 "recall, fit part": fit_scores["recall"],
                 "recall, validation": valid_scores["recall"],
                 "precision, validation": valid_scores["precision"],
                 "accuracy, validation": valid_scores["accuracy"]})
sweep = pd.DataFrame(rows)
sweep

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 3.8))
ax.plot(sweep["depth"], sweep["recall, validation"], marker="o", label="recall: late answers found")
ax.plot(sweep["depth"], sweep["precision, validation"], marker="s", linestyle="--",
        label="precision: alarms that are real")
ax.set_ylim(0, 1)
ax.set_xlabel("depth the tree is allowed")
ax.set_ylabel("score on the validation part")
ax.set_title("No depth wins on both: more recall is paid for in precision")
ax.legend()
fig.tight_layout()

There is no single best row, and that is the honest finding. Depth 2 is the
most careful: 86 % of its alarms are real, and it finds a fifth of the late
answers. Depth 20 finds more than twice as many, and a third of its alarms are
false. Which is "best" depends on what a false alarm costs (a needless phone
call) against what a miss costs (a consumer left waiting), and that is a
decision for the agency, not for `max_depth`.

Overfitting is in the table too. At depth 20 the tree finds 57 % of the late
answers among the rows it was fitted on and 44 % among rows it has not seen.

Accuracy, the last column, hardly moves at all. Choosing by accuracy here
would be choosing by the third decimal of a number that does not measure the
job.

Say the agency can make about 3,000 calls for every 320,000 complaints, and
wants as many of them as possible to be real. That points to a shallow tree.
Depth 4 it is, measured **once** on the test set:

In [ ]:
final = DecisionTreeClassifier(max_depth=4, random_state=SEED).fit(X_train_2, y_train)
final_scores = scores(y_test, final.predict(X_test_2))
print({name: round(value, 4) for name, value in final_scores.items()})

---

## Step 8 — Will it work next year?

The split so far was random: complaints from 2013 and from 2019 are mixed in
both parts. The model was tested on complaints *from the same years it learned
from*. That is not the situation it will be used in. In use, it learns from
the past and predicts the future.

So test it that way: learn from everything up to 2017, predict 2018 and 2019.

In [ ]:
past = complaints["year"] <= 2017

past_history = complaints[past].groupby("company")["late"].agg(late_share="mean", complaints="size")
past_overall = complaints.loc[past, "late"].mean()


def with_past_history(mask):
    company = complaints.loc[mask, "company"]
    added = X[mask].copy()
    added["company_late_share"] = company.map(past_history["late_share"]).fillna(past_overall).to_numpy()
    added["company_complaints"] = company.map(past_history["complaints"]).fillna(0).to_numpy()
    return added


X_past, X_future = with_past_history(past), with_past_history(~past)
y_past, y_future = y[past], y[~past]

forward = DecisionTreeClassifier(max_depth=4, random_state=SEED).fit(X_past, y_past)
future_scores = scores(y_future, forward.predict(X_future))

print("random split:     ", {k: round(v, 4) for k, v in final_scores.items()})
print("past -> future:   ", {k: round(v, 4) for k, v in future_scores.items()})
print("baseline, future: {:.4f}".format((y_future == 0).mean()))

On the future, precision drops: fewer of the alarms are real. Companies
change. Some that used to answer late have improved or closed, and new ones
with no history have appeared. A model trained on the past carries the past's
reputations forward.

The random split was optimistic because it let the model see complaints from
2018 while predicting others from 2018. Whenever the real use is "predict what
comes next", the honest test set is the *later* part of the data, not a random
part. Which split is right is decided by how the model will be used.

---

## Step 9 — What to report

What an honest statement of the result includes:

> The task was to flag, on arrival, complaints that will be answered late
> (2.5 % of 1,282,355 complaints, 2011 to 2019). Always answering "on time" is
> 97.5 % accurate, so accuracy is not a useful score here; recall and
> precision are reported. Using the product, the channel and the date alone, a
> decision tree finds none of the late answers. Adding each company's past
> share of late answers, computed from training data only, a tree of depth 4
> finds roughly three late answers in ten, and about seven of every ten
> complaints it flags are indeed answered late. Trained on 2011 to 2017 and
> tested on 2018 and 2019, its precision is lower, about five in ten.
> The response category was excluded because it records the outcome. The model
> reflects how companies behaved in the past and says nothing about why.

### The traps, in the order they appeared

| Step | Trap | How it was caught |
|---|---|---|
| 3 | An impressive accuracy | comparing it with the baseline |
| 4 | A model that never predicts the rare class | the confusion matrix; recall |
| 5 | A feature that is the answer in disguise | a score too good to be true; reading the tree |
| 6 | A feature computed with the test rows in it | building it from training rows only |
| 7 | Choosing by a score that does not measure the job | asking what a false alarm and a miss cost |
| 8 | Testing on the same period the model learned from | splitting by time |

None of the six raised an error. Every one produced a number, and every number
looked fine.

---

## Exercises

### 🧩 Exercise 1 — The other baseline

"Always on time" has recall 0. What are the accuracy, precision and recall of
the opposite guess, "always late", on the test set? And of flagging a random
2.5 % of complaints (`np.random.default_rng(1).random(len(y_test)) < 0.025`)?
The model of Step 6 should be compared with that last one: how many times more
precise is it than random flagging?

### 🧩 Exercise 2 — A simpler rule

Skip the tree. Flag every complaint whose `company_late_share` (from the
training set) is above 0.3. Compute precision and recall on the test set, and
compare with the tree of depth 4. What did the tree add?

### 🧩 Exercise 3 — The strict validation

Step 7 reused a company history built from the whole training set. Do it
strictly: split the training set into a fit part and a validation part
*first*, build the history from the fit part only, apply it to both, and redo
the sweep. Do the validation scores go down? By how much?

### 🧩 Exercise 4 — Small companies

A company with two training complaints, one late, has a late share of 0.5 and
is flagged. Add a feature that is the late share only for companies with at
least 20 training complaints, and the overall share otherwise. Does precision
improve? Session 9 and Session 10 both met this problem: a share computed from
a handful of cases.

### 🧩 Exercise 5 — Another target

The file has a column "Consumer disputed?" (Yes, No, or missing after 2017).
Take the complaints where it is not missing and try to predict a dispute from
what is known on arrival. Compute the baseline first. Report what you find
even if, and especially if, the model does not beat it.

---

**Next:** `03_Machine_Learning_Reference` lists the tools used here, by task.
Homework 12 has a gentler target (a third of the restaurants are closed), a
tree that does beat its baseline by a few points, and a bigger tree that falls
below it.